# Part 3 — Advanced Modeling: Ensembles, Tuning and ML Pipeline

Building ensemble models, comparing performance, tuning hyperparameters, and saving the best pipeline.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.pipeline import make_pipeline

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score
)

import joblib

## Load Dataset

Load cleaned dataset from Part 1.

In [2]:
df = pd.read_csv(
    "../Part-1-Data-Acquisition-and-EDA/cleaned_data.csv"
)

df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


## Create Classification Dataset

Create binary target using median SalePrice.

In [3]:
# Target

y_clf = (
    df["SalePrice"] > df["SalePrice"].median()
).astype(int)


# Features

X = df.drop(
    "SalePrice",
    axis=1
)


print(X.shape)
print(y_clf.value_counts())

(2930, 81)
SalePrice
0    1467
1    1463
Name: count, dtype: int64


## Encode Categorical Features

In [4]:
categorical_cols = X.select_dtypes(
    include="object"
).columns


X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
)


X = X.astype(float)


print("Final shape:", X.shape)

Final shape: (2930, 262)


C:\Users\palla\AppData\Local\Temp\ipykernel_24488\2054963072.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


## Train-Test Split and Scaling

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_clf,
    test_size=0.2,
    random_state=42
)


scaler = StandardScaler()


X_train_scaled = scaler.fit_transform(
    X_train
)


X_test_scaled = scaler.transform(
    X_test
)


print(X_train_scaled.shape)
print(X_test_scaled.shape)

(2344, 262)
(586, 262)


## Decision Tree Baseline

A default Decision Tree is trained to compare training and testing performance.

In [6]:
# Default Decision Tree

dt_model = DecisionTreeClassifier(
    random_state=42
)


dt_model.fit(
    X_train_scaled,
    y_train
)


# Predictions

train_pred = dt_model.predict(X_train_scaled)
test_pred = dt_model.predict(X_test_scaled)


# Accuracy

dt_train_acc = accuracy_score(
    y_train,
    train_pred
)

dt_test_acc = accuracy_score(
    y_test,
    test_pred
)


print("Decision Tree Baseline")
print("----------------------")
print("Training Accuracy:", dt_train_acc)
print("Testing Accuracy:", dt_test_acc)

Decision Tree Baseline
----------------------
Training Accuracy: 1.0
Testing Accuracy: 0.909556313993174


## Controlled Decision Tree

A limited-depth tree is trained to reduce overfitting.

In [7]:
# Controlled Decision Tree

dt_controlled = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)


dt_controlled.fit(
    X_train_scaled,
    y_train
)


# Predictions

train_pred_control = dt_controlled.predict(
    X_train_scaled
)


test_pred_control = dt_controlled.predict(
    X_test_scaled
)


# Accuracy

control_train_acc = accuracy_score(
    y_train,
    train_pred_control
)


control_test_acc = accuracy_score(
    y_test,
    test_pred_control
)


print("Controlled Decision Tree")
print("------------------------")
print("Training Accuracy:", control_train_acc)
print("Testing Accuracy:", control_test_acc)

Controlled Decision Tree
------------------------
Training Accuracy: 0.9249146757679181
Testing Accuracy: 0.9163822525597269


## Gini vs Entropy Comparison

Decision Trees are trained using two splitting criteria.

In [8]:
# Gini Tree

dt_gini = DecisionTreeClassifier(
    max_depth=5,
    criterion="gini",
    random_state=42
)


dt_gini.fit(
    X_train_scaled,
    y_train
)


gini_pred = dt_gini.predict(
    X_test_scaled
)


gini_acc = accuracy_score(
    y_test,
    gini_pred
)



# Entropy Tree

dt_entropy = DecisionTreeClassifier(
    max_depth=5,
    criterion="entropy",
    random_state=42
)


dt_entropy.fit(
    X_train_scaled,
    y_train
)


entropy_pred = dt_entropy.predict(
    X_test_scaled
)


entropy_acc = accuracy_score(
    y_test,
    entropy_pred
)



print("Gini Accuracy:", gini_acc)
print("Entropy Accuracy:", entropy_acc)

Gini Accuracy: 0.9215017064846417
Entropy Accuracy: 0.9215017064846417


## Random Forest Classifier

Random Forest combines multiple decision trees to reduce variance and improve generalization.

In [9]:
# Random Forest model

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)


rf_model.fit(
    X_train_scaled,
    y_train
)


# Predictions

rf_train_pred = rf_model.predict(
    X_train_scaled
)


rf_test_pred = rf_model.predict(
    X_test_scaled
)


rf_prob = rf_model.predict_proba(
    X_test_scaled
)[:,1]


# Accuracy

rf_train_acc = accuracy_score(
    y_train,
    rf_train_pred
)


rf_test_acc = accuracy_score(
    y_test,
    rf_test_pred
)


# AUC

rf_auc = roc_auc_score(
    y_test,
    rf_prob
)


print("Random Forest Results")
print("---------------------")
print("Training Accuracy:", rf_train_acc)
print("Testing Accuracy:", rf_test_acc)
print("ROC-AUC:", rf_auc)

Random Forest Results
---------------------
Training Accuracy: 0.9867747440273038
Testing Accuracy: 0.9488054607508533
ROC-AUC: 0.9905139723470042


## Random Forest Feature Importance

The top features are identified using feature importance scores.

In [10]:
# Feature importance dataframe

importance_df = pd.DataFrame({

    "Feature": X.columns,

    "Importance": rf_model.feature_importances_

})


importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)


importance_df.head(5)

,Feature,Importance
17,Gr Liv Area,0.082375
7,Year Built,0.074415
5,Overall Qual,0.058056
20,Full Bath,0.049611
27,Garage Cars,0.049173


## Gradient Boosting Classifier

Gradient Boosting builds trees sequentially, where each new tree improves errors made by previous trees.

In [11]:
from sklearn.ensemble import GradientBoostingClassifier


# Gradient Boosting model

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)


gb_model.fit(
    X_train_scaled,
    y_train
)


# Predictions

gb_train_pred = gb_model.predict(
    X_train_scaled
)


gb_test_pred = gb_model.predict(
    X_test_scaled
)


gb_prob = gb_model.predict_proba(
    X_test_scaled
)[:,1]


# Metrics

gb_train_acc = accuracy_score(
    y_train,
    gb_train_pred
)


gb_test_acc = accuracy_score(
    y_test,
    gb_test_pred
)


gb_auc = roc_auc_score(
    y_test,
    gb_prob
)


print("Gradient Boosting Results")
print("-------------------------")
print("Training Accuracy:", gb_train_acc)
print("Testing Accuracy:", gb_test_acc)
print("ROC-AUC:", gb_auc)

Gradient Boosting Results
-------------------------
Training Accuracy: 0.9748293515358362
Testing Accuracy: 0.9402730375426621
ROC-AUC: 0.9901639344262295


## Feature Ablation Study

The least important features from Random Forest are removed to evaluate their impact on model performance.

In [12]:
# Find bottom 5 features

lowest_features = importance_df.tail(5)

lowest_features

,Feature,Importance
208,Kitchen Qual_Po,0.0
40,MS Zoning_I (all),0.0
230,Garage Qual_Po,0.0
234,Garage Cond_Po,0.0
257,Sale Condition_AdjLand,0.0


## Random Forest Feature Ablation

The model is retrained after removing the five least important features.

In [13]:
# Remove lowest importance features

remove_features = lowest_features["Feature"].tolist()


X_train_reduced = X_train.drop(
    columns=remove_features
)


X_test_reduced = X_test.drop(
    columns=remove_features
)


print("Original Features:", X_train.shape[1])
print("Reduced Features:", X_train_reduced.shape[1])

Original Features: 262
Reduced Features: 257


In [14]:
# Scale reduced features

scaler_reduced = StandardScaler()


X_train_reduced_scaled = scaler_reduced.fit_transform(
    X_train_reduced
)


X_test_reduced_scaled = scaler_reduced.transform(
    X_test_reduced
)


# Train reduced Random Forest

rf_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)


rf_reduced.fit(
    X_train_reduced_scaled,
    y_train
)


# Probability prediction

rf_reduced_prob = rf_reduced.predict_proba(
    X_test_reduced_scaled
)[:,1]


# AUC

reduced_auc = roc_auc_score(
    y_test,
    rf_reduced_prob
)


print("Full Model AUC:", rf_auc)
print("Reduced Model AUC:", reduced_auc)

Full Model AUC: 0.9905139723470042
Reduced Model AUC: 0.9912373840499387


## Cross Validation Comparison

5-fold Stratified Cross Validation is used to compare model generalization performance.

In [15]:
from sklearn.linear_model import LogisticRegression


# Models

models = {

    "Logistic Regression": LogisticRegression(
        max_iter=1000
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=20,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


cv_results = []


for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train_scaled,
        y_train,
        cv=cv,
        scoring="roc_auc"
    )


    cv_results.append({

        "Model": name,

        "Mean AUC": scores.mean(),

        "Std AUC": scores.std()

    })


cv_df = pd.DataFrame(cv_results)


cv_df

,Model,Mean AUC,Std AUC
0,Logistic Regression,0.965608,0.004143
1,Decision Tree,0.930523,0.003047
2,Random Forest,0.979915,0.004128
3,Gradient Boosting,0.982125,0.003290


## Hyperparameter Tuning with GridSearchCV

A pipeline is created to combine preprocessing steps and Random Forest training.

In [17]:
from sklearn.pipeline import Pipeline


pipeline = Pipeline([

    ("imputer", SimpleImputer(strategy="median")),

    ("scaler", StandardScaler()),

    ("randomforestclassifier",
     RandomForestClassifier(
         random_state=42
     ))

])

param_grid = {

    "randomforestclassifier__n_estimators": [
        50,100,200
    ],

    "randomforestclassifier__max_depth": [
        5,10,None
    ],

    "randomforestclassifier__min_samples_leaf": [
        1,5
    ]

}


pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto accou

In [18]:
grid_search = GridSearchCV(

    pipeline,

    param_grid,

    cv=StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ),

    scoring="roc_auc",

    n_jobs=-1

)


grid_search.fit(
    X_train,
    y_train
)


print("Best Parameters:")
print(grid_search.best_params_)


print("\nBest CV AUC:")
print(grid_search.best_score_)

Best Parameters:
{'randomforestclassifier__max_depth': None, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 200}

Best CV AUC:
0.9802635190419965
